## Download JPEGs and Calculate Extents

Prepare all photos in a selected subset for georeferencing.

- Download original JPEGs
- Read image dimensions and DPI
- Calculate approximate horizontal ground extents
- Save preparation results to a separate tracker GeoPackage

Existing valid JPEGs are reused rather than downloaded again.

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm
import shutil

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_horizontal_extent,
    update_tracker,
    ground_to_mercator_extent
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

c:\Users\eu01251115\LocalProjects\batch_georef\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load subset table and make paths

In [2]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# Create a separate GeoPackage for tracking processing progress
TRACKER_PATH = SUBSET_PATH.with_name(
    f"{SUBSET_PATH.stem}_tracker.gpkg"
)

# define subfolder for downloading images, using the geopackage name
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem

### Make tracker geopackage and download directory

In [3]:
# Create tracker gpkg, ONLY if it doesn't already exist
# This is a copy of the subset table, which is preserved blank for input. 

if not TRACKER_PATH.exists():
    shutil.copy2(SUBSET_PATH, TRACKER_PATH)
    print(f"Created tracker: {TRACKER_PATH.name}")
else:
    print(f"Using existing tracker: {TRACKER_PATH.name}")

Using existing tracker: subset_20000_carver_37_38_40_51_57_63_tracker.gpkg


In [4]:
# Make download directory if it doesn't already exist
# exist_ok makes it succeed silently and not do anything if it already exists
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

### Load tracker, select unfinished photos

In [5]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

In [6]:
tracker_gdf

,cty_year_id,status,county,year,photo_id,h_extent,ground_width,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,error,latitude,geometry
0,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,None,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,None,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,None,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,None,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,None,44.775835,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,carver_1963_wm-1dd-111,None,Carver,1963,wm-1dd-111,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,None,None,None,44.702124,POINT (-10428717.028 5574748.276)
867,carver_1963_wm-1dd-113,None,Carver,1963,wm-1dd-113,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,None,None,None,44.702615,POINT (-10424133.337 5574825.175)
868,carver_1963_wm-1dd-228,None,Carver,1963,wm-1dd-228,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,None,None,None,44.673257,POINT (-10435676.055 5570228.328)
869,carver_1963_wm-1dd-230,None,Carver,1963,wm-1dd-230,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,None,None,None,44.673322,POINT (-10431122.308 5570238.503)


In [7]:
# Set to True to re-prepare all photos, regardless of status.
# does not re-download the photo if it exists. to do that, delete the files
REPREPARE_ALL = False

if REPREPARE_ALL:
    photos_gdf = tracker_gdf.copy()
else:
    photos_gdf = tracker_gdf.loc[
        ~tracker_gdf["status"].isin(["prepared", "georeferenced"])
    ].copy()

print(f"Total photos in subset: {len(tracker_gdf)}")
print(f"Photos to prepare: {len(photos_gdf)}")
print(f"Download directory: {DOWNLOAD_DIR}")

display(photos_gdf.head())

Total photos in subset: 871
Photos to prepare: 871
Download directory: data\downloads\subset_20000_carver_37_38_40_51_57_63


,cty_year_id,status,county,year,photo_id,h_extent,ground_width,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,error,latitude,geometry
0,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,None,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,None,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,None,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,None,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,None,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,None,44.775835,POINT (-10464934.045 5586300.061)


### Download JPEG, read metadata, calculate extent

In [8]:
# create an empty list to collect results for each photo. 
results = []

# loop through photos_gdf, processing one photo (row) at a time. 
for _, photo in tqdm(
    photos_gdf.iterrows(),
    total=len(photos_gdf),
    desc="Preparing photos"
):
    # get the unique but friendly photo identifier
    photo_id = photo["cty_year_id"]

    # and use it to construct the download path
    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    # Create a dictionary to store this photo's processing results.
    # Default values are replaced as each step succeeds.
    result = {
        "source_id": photo["source_id"],
        "width_px": None,
        "dpi_x": None,
        "ground_width": None,
        "h_extent": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # download JPEG or reuse if already downloaded
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        # record download success and date

        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

        # read metadata from the downloaded image
        metadata = get_image_metadata(image_path)

        # update that dictionary
        result.update(metadata)

        # estimate the photos horizontal ground coverage in meters
        result["ground_width"] = calculate_horizontal_extent(
            width_px=metadata["width_px"],
            dpi_x=metadata["dpi_x"],
            scale=photo["scale"],
        )

        # add web mercator distortion to ground width, based on latitude
        result["h_extent"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude = photo["latitude"],
        )

    except Exception as e:
        # If any step fails, record the error and continue
        result["error"] = str(e)

    # Save this photo's result dict to the list of dicts
    results.append(result)

Preparing photos: 100%|██████████| 871/871 [00:01<00:00, 704.90it/s]


In [9]:
# convert list of results dicts into a dataframe
results_df = pd.DataFrame(results)

In [13]:
# examine
pd.set_option('display.max_rows', 40)

results_df

,source_id,width_px,dpi_x,ground_width,h_extent,date_prepared,error
0,91674,7105,800.0,4511.675000,6350.576497,2026-11-04,NaN
1,91675,7107,800.0,4512.945000,6353.632922,2026-11-04,NaN
2,91676,7174,800.0,4555.490000,6414.812208,2026-11-04,NaN
3,91677,7110,800.0,4514.850000,6358.855858,2026-11-04,NaN
4,91678,7038,800.0,4469.130000,6295.721017,2026-11-04,NaN
...,...,...,...,...,...,...,...
866,92355,5453,600.0,4616.873333,6495.562927,2026-11-04,NaN
867,92356,5477,600.0,4637.193333,6524.206835,2026-11-04,NaN
868,92362,5487,600.0,4645.660000,6532.806900,2026-11-04,NaN
869,92363,5479,600.0,4638.886667,6523.289439,2026-11-04,NaN


In [11]:
results_df[["ground_width", "h_extent", "error"]].notna().sum()

ground_width    713
h_extent        713
error           158
dtype: int64

In [14]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

Updated 871 records.


In [15]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,h_extent,ground_width,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,error,latitude,geometry
0,carver_1937_wm-26-2368,prepared,Carver,1937,wm-26-2368,6350.57649697092,4511.674999999999,None,None,20000,800.0,7105,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,2026-11-04,None,NaN,44.729640,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,prepared,Carver,1937,wm-26-2369,6353.632921854273,4512.945,None,None,20000,800.0,7107,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,2026-11-04,None,NaN,44.741189,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,prepared,Carver,1937,wm-26-2370,6414.812208153059,4555.489999999999,None,None,20000,800.0,7174,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,2026-11-04,None,NaN,44.752738,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,prepared,Carver,1937,wm-26-2371,6358.855857537255,4514.849999999999,None,None,20000,800.0,7110,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,2026-11-04,None,NaN,44.764287,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,prepared,Carver,1937,wm-26-2372,6295.721017087898,4469.129999999999,None,None,20000,800.0,7038,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,2026-11-04,None,NaN,44.775835,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,carver_1963_wm-1dd-111,prepared,Carver,1963,wm-1dd-111,6495.56292673825,4616.873333333333,None,None,20000,600.0,5453,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,2026-11-04,None,NaN,44.702124,POINT (-10428717.028 5574748.276)
867,carver_1963_wm-1dd-113,prepared,Carver,1963,wm-1dd-113,6524.2068352573315,4637.193333333334,None,None,20000,600.0,5477,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,2026-11-04,None,NaN,44.702615,POINT (-10424133.337 5574825.175)
868,carver_1963_wm-1dd-228,prepared,Carver,1963,wm-1dd-228,6532.80690046775,4645.66,None,None,20000,600.0,5487,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,2026-11-04,None,NaN,44.673257,POINT (-10435676.055 5570228.328)
869,carver_1963_wm-1dd-230,prepared,Carver,1963,wm-1dd-230,6523.28943931267,4638.886666666666,None,None,20000,600.0,5479,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,2026-11-04,None,NaN,44.673322,POINT (-10431122.308 5570238.503)
